# Assessment of the reference energy system (2023)

In [1]:
# %pip install brightway2
# %pip install mescal
# %pip install energyscope

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
LCA_PROJECTS_ROOT = NOTEBOOK_DIR.parent.parent
sys.path.insert(1, str(LCA_PROJECTS_ROOT / '00_Shared'))

In [4]:
from utils import (
    COMMON_DATA_DIR,
    N_capita_2023,
    wood_list, wet_biomass_list, waste_list,
    get_impact_scores,
    default_colors_sankey,
)
from new_plots import _create_sankey_figure, generate_sankey_flows

In [5]:
import os
import pandas as pd
import numpy as np
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from ast import literal_eval
import plotly.express as px
import plotly.graph_objects as go
import bw2data as bd
from tqdm import tqdm
from shared.utils import run_model, load_snapshot

In [6]:
import plotly.io as pio
pio.renderers.default = "png"

In [7]:
save_results = False
reference_year = 2023

In [8]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [9]:
LCA_RESULTS_DIR = f'../03_Results/LCA/{reference_year}'

In [10]:
bd.projects.set_current('ecoinvent3.12')

In [11]:
es_tech_df = pd.read_csv(COMMON_DATA_DIR / 'technology_dictionary.csv')

In [12]:
# Create a dict from the Programming Name and Long name columns of es_tech_dict
es_tech_name_dict = dict(zip(es_tech_df['Programming name'], es_tech_df['Long name']))
es_tech_name_dict = {k: v for k, v in es_tech_name_dict.items() if k not in wood_list+wet_biomass_list+waste_list}

## Initialize the model

In [13]:
# Initialize the reference QC model with .mod and .dat files
model = load_snapshot(reference_year)

In [14]:
# Solve the model and get results
results = run_model(model)

Gurobi 12.0.0: 

In [15]:
df_sankey = generate_sankey_flows(
        results=results,
        aggregate_mobility=True,
        aggregate_grid=True,
        aggregate_technology=True,
        run_id=0,
    )
df_sankey['source (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['source']] if x['source'] in es_tech_name_dict else x['source'], axis=1)
df_sankey['target (long)'] = df_sankey.apply(lambda x: es_tech_name_dict[x['target']] if x['target'] in es_tech_name_dict else x['target'], axis=1)

fig = _create_sankey_figure(df_sankey, colors=default_colors_sankey, long_names=True)

if save_results:
    fig.write_html(f'../03_Results/Figures/reference/sankey_{reference_year}.html')

if save_results:
    df_sankey.to_csv(f'../03_Results/Tables/reference/sankey_raw_{reference_year}.csv', index=False)
df_sankey['value'] *= 1e-3 # from GWh to TWh

## Impact assessment

In [33]:
impact_categories_list = [
    i for i in bd.methods if
    (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12')
    | (i[0] == 'IMPACT World+ Damage 2.2.1 for ecoinvent v3.12 (incl. CO2 uptake)')
]

impact_categories_list +=[
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Remaining ecosystem quality'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Remaining human health'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Ecosystem quality', 'Total ecosystem quality (biogenic)'),
    ('IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12', 'Human health', 'Total human health (biogenic)'),
]

impact_categories_list = [i for i in impact_categories_list if i[2] not in ['Total ecosystem quality', 'Total human health']]
impact_categories_list = [i for i in impact_categories_list if  # keep only climate change and marine acidification from -1/+1 version of IW+
    not (i[0] == 'IMPACT World+ Damage 2.2.1_regionalized for ecoinvent v3.12' and ('Marine acidification' in i[2] or 'Climate change' in i[2]))
]

impact_categories_list = [i for i in impact_categories_list if not ('Climate change' in i[2] and 'total' not in i[2])]  # keep only climate change total

In [34]:
scenarios_list = [
    {"model": None, "pathway": None, "year": 2023}, # +1.7°C
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [63]:
comparison_2023_2050 = []

for scenario in scenarios_list:

    year = scenario['year']
    model = scenario['model']
    pathway = scenario['pathway']
    if year == 2050:
        path_lca_results = f'../03_Results/LCA/{year}/{model}/{pathway}/'
    else:
        path_lca_results = f'../03_Results/LCA/{year}/'

    # Loading LCA results
    impact_scores = pd.read_csv(path_lca_results+'impact_scores.csv')
    impact_scores_direct = pd.read_csv(path_lca_results+'impact_scores_direct_emissions.csv')

    # from [kg CO2-eq / kW(h) or pkm(/h) or tkm(/h)] to [t-CO2-eq / GW(h) or Mpkm(/h) or Mtkm(/h)]
    impact_scores.Value *= 1e3
    impact_scores_direct.Value *= 1e3

    # Reading impact categories as tuples
    impact_scores.Impact_category = impact_scores.Impact_category.apply(lambda x: literal_eval(x))
    impact_scores_direct.Impact_category = impact_scores_direct.Impact_category.apply(lambda x: literal_eval(x))

    # Merging impact scores with energy configuration results
    df_f_mult, df_annual_prod, df_annual_res = get_impact_scores(
        impact_category=impact_categories_list,
        df_impact_scores=impact_scores,
        df_results=results,
    )

    df_annual_prod_direct = get_impact_scores(
        impact_category=impact_categories_list,
        df_impact_scores=impact_scores_direct,
        df_results=results,
        assessment_type='direct',
    )

    impact_tot_tthh = df_f_mult['Total human health (biogenic)'].sum() + df_annual_prod['Total human health (biogenic)'].sum() + df_annual_res['Total human health (biogenic)'].sum()
    impact_tot_tteq = df_f_mult['Total ecosystem quality (biogenic)'].sum() + df_annual_prod['Total ecosystem quality (biogenic)'].sum() + df_annual_res['Total ecosystem quality (biogenic)'].sum()

    for cat in impact_categories_list:

        impact_constr = df_f_mult[cat[-1]].sum()
        impact_op = df_annual_prod[cat[-1]].sum()
        impact_op_direct = df_annual_prod_direct[cat[-1]].sum()
        impact_res_wo_biomass = df_annual_res[~df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()
        impact_res_biomass = df_annual_res[df_annual_res['index'].isin(wood_list+wet_biomass_list+waste_list)][cat[-1]].sum()

        impact_tot = impact_constr + impact_op + impact_res_wo_biomass + impact_res_biomass
        impact_op_indirect = impact_op - impact_op_direct

        contrib_to_total_aop = 100 * impact_tot / (impact_tot_tthh if cat[1] == 'Human health' else impact_tot_tteq)

        comparison_2023_2050.append([
            cat[-1],
            year,
            model,
            pathway,
            impact_constr,
            impact_op_direct,
            impact_op_indirect,
            impact_res_wo_biomass,
            impact_res_biomass,
            impact_tot,
            contrib_to_total_aop,
        ])

comparison_2023_2050 = pd.DataFrame(
    comparison_2023_2050,
    columns=[
        'Impact category', 'Year', 'IAM', 'SSP-RCP',
        'Construction',
        'Operation (direct)', 'Operation (indirect)',
        'Resources (wo biomass)', 'Resources (biomass)',
        'Total',
        'Contribution to total AoP (%)',
    ]).set_index(['Impact category', 'Year', 'IAM', 'SSP-RCP'])

In [65]:
,
adjustment_ratios = []

for scenario in scenarios_list:
    year = scenario['year']
    model = scenario['model']
    pathway = scenario['pathway']

    if year == 2023:
        continue

    for cat in [i[2] for i in impact_categories_list]:
        tot_2050 = comparison_2023_2050.loc[cat].loc[year].loc[model].loc[pathway]['Total']
        tot_2023 = comparison_2023_2050.loc[cat].loc[2023].loc[np.nan].loc[np.nan]['Total']
        adjustment_ratios.append([cat, model, pathway, tot_2050 / tot_2023])

In [68]:
df_adjustment_ratios = pd.DataFrame(adjustment_ratios, columns=['Impact category', 'IAM', 'SSP-RCP', 'Ratio']).set_index(['Impact category', 'IAM', 'SSP-RCP'])
df_adjustment_ratios = pd.merge(
    comparison_2023_2050,
    df_adjustment_ratios,
    how='left',
    left_index=True,
    right_index=True,
)

In [69]:
df_adjustment_ratios

,,,,Construction,Operation (direct),Operation (indirect),Resources (wo biomass),Resources (biomass),Total,Contribution to total AoP (%),Ratio
Impact category,Year,IAM,SSP-RCP,,,,,,,,
Fisheries impact,2023,NaN,NaN,3.060314e-09,0.000000e+00,6.993923e-12,1.928130e-12,3.590005e-13,3.069595e-09,7.166538e-16,NaN
Freshwater acidification,2023,NaN,NaN,2.258242e+05,1.722672e+06,5.736933e+04,1.537677e+05,1.782419e+03,2.161416e+06,5.046225e-01,NaN
"Freshwater ecotoxicity, long term",2023,NaN,NaN,2.157542e+06,1.649981e+05,2.386418e+06,1.265106e+06,1.982177e+04,5.993886e+06,1.399384e+00,NaN
"Freshwater ecotoxicity, short term",2023,NaN,NaN,4.900845e+05,9.178466e+04,3.776592e+05,1.529157e+06,8.475230e+03,2.497160e+06,5.830083e-01,NaN
Freshwater eutrophication,2023,NaN,NaN,1.530206e+03,0.000000e+00,4.340955e+02,5.909004e+02,6.052584e+00,2.561254e+03,5.979722e-04,NaN
...,...,...,...,...,...,...,...,...,...,...,...
"Climate change, human health, long term, total",2050,image,SSP1-L,2.832873e+01,3.912830e+02,1.652487e+01,1.558581e+01,-6.049362e+01,3.912288e+02,4.195331e+01,0.749379
Remaining ecosystem quality,2050,image,SSP1-L,7.110329e+06,2.485282e+07,7.705048e+06,1.036066e+07,-1.730107e+06,4.829875e+07,1.460756e+01,0.927858
Remaining human health,2050,image,SSP1-L,1.897471e+01,3.712381e+02,1.499883e+01,1.271404e+01,2.283382e-01,4.181541e+02,4.484063e+01,0.893574


In [ ]:
if save_results:
    df_adjustment_ratios.to_csv('../03_Results/Tables/reference/adjustment_ratios.csv', index=False)